# 03 — 4-Class Extension (Contribution 1)
**DeepLense GSoC 2026 — Pallab Mondal**

This notebook:
1. Trains LensPINN extended to **4 classes** (adds *vortex* dark matter)
2. Benchmarks against the 3-class LensPINN baseline
3. Runs GradCAM to verify the model attends to physically meaningful regions (Einstein ring / arcs)

In [ ]:
import sys, os
from pathlib import Path

MY_WORK = Path(os.getcwd()).parent
sys.path.insert(0, str(MY_WORK))

import torch
import numpy as np
import matplotlib.pyplot as plt

from config import (
    MODEL_I_TRAIN, MODEL_I_TEST, CLASS_NAMES,
    IMAGE_SIZE, BATCH_SIZE, CHECKPOINTS_DIR, PLOTS_DIR,
)
from utils.data_loader    import build_dataloaders
from utils.metrics        import evaluate_model, per_class_auc, plot_confusion_matrix
from models.lens_pinn     import LensPINN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── 1.  Build dataloaders (3-class for now until vortex data is available) ────
# NOTE: When vortex training data becomes available, change class_names
# to CLASS_NAMES (all 4) and point at a dataset that contains it.

train_loader, val_loader, test_loader = build_dataloaders(
    train_root=MODEL_I_TRAIN,
    test_root=MODEL_I_TEST,
    class_names=CLASS_NAMES[:3],   # will be 4 when vortex data arrives
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')
print(f'Test  batches: {len(test_loader)}')

In [ ]:
# ── 2.  Instantiate 4-class LensPINN ─────────────────────────────────────────
model = LensPINN(
    num_classes=3,    # change to 4 when vortex data is ready
    vit_name='vit_small_patch16_224',
    feature_dim=512,
    dropout=0.3,
    theta_E_max=2.0,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'LensPINN trainable parameters: {n_params:,}')

In [ ]:
# ── 3.  Quick training (use train.py for full runs) ───────────────────────────
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=5)  # short demo run

DEMO_EPOCHS = 2   # set to 100 for a real training run
LAMBDA_PHYS = 0.1

history = {'train_loss': [], 'val_acc': []}

for epoch in range(1, DEMO_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    n = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        theta_E, source, logits = model(imgs)
        cls_loss  = criterion(logits, labels)
        phys_loss = source.abs().mean()  # sparsity prior
        loss = cls_loss + LAMBDA_PHYS * phys_loss
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)

    avg_loss = running_loss / n

    # Val accuracy
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, _, logits = model(imgs)
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)

    val_acc = correct / total
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch}/{DEMO_EPOCHS}  train_loss={avg_loss:.4f}  val_acc={val_acc:.4f}')
    scheduler.step()

In [ ]:
# ── 4.  Evaluation ───────────────────────────────────────────────────────────
results = evaluate_model(
    model, test_loader, device,
    class_names=CLASS_NAMES[:3],
    verbose=True,
)

print('\nPer-class AUC:')
per_class_auc(model, test_loader, device, CLASS_NAMES[:3])

In [ ]:
# ── 5.  Confusion matrix ──────────────────────────────────────────────────────
plot_confusion_matrix(
    model, test_loader, device,
    class_names=CLASS_NAMES[:3],
    save_path=str(PLOTS_DIR / '03_lens_pinn_confusion.png'),
)

In [ ]:
# ── 6.  GradCAM — does the model attend to the Einstein ring? ─────────────────
from utils.metrics import gradcam_visualise

# Pick one image per class and visualise GradCAM
from utils.data_loader import DeepLenseDataset, get_val_transform

test_ds = DeepLenseDataset(
    root=MODEL_I_TEST,
    class_names=CLASS_NAMES[:3],
    transform=get_val_transform(IMAGE_SIZE),
)

target_layer = model.cnn_source.net[-3]  # last Conv2d in source branch

for cls_name in CLASS_NAMES[:3]:
    cls_idx = CLASS_NAMES.index(cls_name)
    sample_idx = next(i for i, (_, l) in enumerate(test_ds.samples) if l == cls_idx)
    img, label = test_ds[sample_idx]

    print(f'\nGradCAM for class: {cls_name}')
    gradcam_visualise(
        model=model,
        image=img.unsqueeze(0).to(device),
        target_layer=target_layer,
        class_idx=cls_idx,
        save_path=str(PLOTS_DIR / f'03_gradcam_{cls_name}.png'),
    )

In [ ]:
# ── 7.  Einstein radius distribution ─────────────────────────────────────────
model.eval()
theta_E_vals = {cls: [] for cls in CLASS_NAMES[:3]}

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        theta_E, _, _ = model(imgs)
        for er, lab in zip(theta_E.squeeze().cpu().tolist(), labels.tolist()):
            theta_E_vals[CLASS_NAMES[lab]].append(er)

fig, ax = plt.subplots(figsize=(8, 4))
for cls, vals in theta_E_vals.items():
    ax.hist(vals, bins=40, alpha=0.6, label=cls, density=True)
ax.set(title='Predicted Einstein Radius Distribution per Class',
       xlabel='θ_E (arcsec)', ylabel='Density')
ax.legend()
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '03_einstein_radius_dist.png'), dpi=150)
plt.show()